# Exercise 2.3 — Exploring and cleaning messy patient data

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day2/day2-02-cleaning-encoding.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

## Overview

Real data is messy! If done properly, most of the time in a data anlysis project is spent understanding the data, cleaning it and preprocessing it to the format you require for downstream analysis. Messiness comes in many different ways so just be aware that you need to visualize, ask questions and practice! Below you have some common examples you might encounter as well as some inisght into what one-hot encoding means. 

## The data

`messy_gp_patients.csv` holds 30 patients from a handful of GP practices, with
their age, height and smoking status.

[**Download `messy_gp_patients.csv`**](data/messy_gp_patients.csv) and save it in your `data` folder,
or take every file at once from the zip on the [introduction page](index.qmd).

### 1. First look at the data

Load it and look before you touch anything.

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

gp_py = pd.read_csv("data/messy_gp_patients.csv",
                    keep_default_na=False, na_values=[""])

print(gp_py.shape)
print(gp_py.dtypes)
print(gp_py.describe(include="all"))

> **Why those two arguments**
>
> `pandas` treats `N/A`, `NA`, `null` and `NaN` as missing the moment it reads a
> file. Here that would quietly convert one of the smoking-status codes before you
> ever saw it, and spotting those codes is part of the exercise.
> `keep_default_na=False` switches that off; `na_values=[""]` keeps genuinely
> empty cells missing.

> **Think about it**
>
> 1. How many patients and how many variables are there?
> 2. Which columns came in as numbers and which as text?
> 3. Does anything look wrong with `height_cm` or `age`?

------------------------------------------------------------------------

### 2. Cleaning the numerical variables

#### Height

Look at what is actually in the column:

In [ ]:
print(gp_py["height_cm"].unique())

Two values are not ordinary heights in centimetres: `unknown`, and `1.72` —
a height recorded in metres. They need different treatment, because one of them
is recoverable and the other is not.

In [ ]:
gp_py["height_cm"] = pd.to_numeric(
    gp_py["height_cm"].replace("unknown", np.nan), errors="coerce")

# Assume positive heights below 3 were recorded in metres
gp_py.loc[gp_py["height_cm"] < 3, "height_cm"] *= 100

print(gp_py["height_cm"].describe().round(1))

Heights now run from 155 to 183 cm, with one missing.

In pandas you create a new column, or change an existing one, by assigning to it
(`gp_py["new_column"] = ...`) or, for several at once, with `assign()`. This is
what `mutate()` does in R. In the example above we used it to clean the height
column; you can also create new columns from existing ones. For example, we
could group patients as "short", "average" or "tall" by their height, or just
add an extra column with 1 in every row. More examples are in the
[pandas guide to creating new columns](https://pandas.pydata.org/docs/getting_started/intro_tutorials/05_add_columns.html).

In [ ]:
gp_with_extra_var = gp_py.assign(
    height_cat=pd.cut(gp_py["height_cm"],
                      bins=[-np.inf, 160, 180, np.inf],
                      labels=["short", "average", "tall"]),
    JustOne=1
)

# print(gp_with_extra_var)

Let's continue.

#### Age

Find the implausible ones rather than assuming you know where they are:

In [ ]:
print(gp_py[(gp_py["age"] < 0) | (gp_py["age"] > 110)])

In [ ]:
gp_py["age"] = gp_py["age"].where((gp_py["age"] >= 0) & (gp_py["age"] <= 110))

print(int(gp_py["age"].isna().sum()))

One patient, P014, recorded at 250.

> **Think about it**
>
> Why is replacing an impossible age with missing better than replacing it with
> zero — or than leaving it at 250?

------------------------------------------------------------------------

### 3. Exploring and collapsing the GP practices

How many patients are registered at each practice?

In [ ]:
print(gp_py["GP_practice"].value_counts())

In [ ]:
counts = gp_py["GP_practice"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(counts.index, counts.values, color="steelblue")
ax.set_xlabel("Number of patients")
ax.set_ylabel("GP practice")
fig.tight_layout()
plt.show()

Three practices have a single patient. It is difficult to find any pattern in a
practice with just one patient, so combine anything with fewer than three
patients into an `Other` category. You can do it in many different ways; pandas
has no single function for it (R's `forcats` has `fct_lump_min()`), so we do it
in two steps: count the patients per practice, then replace the rare ones.

In [ ]:
counts = gp_py["GP_practice"].value_counts()
rare = counts[counts < 3].index

gp_py["GP_practice"] = gp_py["GP_practice"].where(
    ~gp_py["GP_practice"].isin(rare), "Other")

print(gp_py["GP_practice"].value_counts())

> **Think about it**
>
> 1. How many practices were there to begin with, and how many are left?
> 2. Where did the three single-patient practices go — and look carefully, because
>    the answer is not "into a new category".
> 3. What information did you just lose?

------------------------------------------------------------------------

### 4. Cleaning and visualising smoking status

Count the categories as recorded:

In [ ]:
print(gp_py["smoking_status"].value_counts(dropna=False))

In [ ]:
counts = gp_py["smoking_status"].value_counts(dropna=False)
counts.index = [str(i) for i in counts.index]
counts = counts.sort_index()

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(counts.index, counts.values, color="steelblue")
ax.set_xlabel("Patients")
ax.set_ylabel("Smoking status")
fig.tight_layout()
plt.show()

Ten different spellings for what are really three categories, plus two codes
that mean "not recorded". Standardise the labels and turn the codes into
missing values:

In [ ]:
spelling = gp_py["smoking_status"].str.strip().str.lower()

labels = {**{k: "Never"   for k in ["never", "non-smoker", "non smoker"]},
          **{k: "Former"  for k in ["former", "ex-smoker", "ex smoker"]},
          **{k: "Current" for k in ["current", "current smoker", "smoker"]},
          **{k: np.nan    for k in ["unknown", "n/a", "", "?"]}}

gp_py["smoking_status"] = spelling.map(labels)

print(gp_py["smoking_status"].value_counts(dropna=False))

Never 12, Former 8, Current 7, and 3 genuinely missing.

> **Think about it**
>
> 1. Why standardise the labels *before* encoding or analysing them?
> 2. `Never` and `never` differed by one capital letter. What would a model have
>    done with them?
> 3. The recipe above lower-cases and trims **before** matching. Why is that order
>    necessary rather than tidy?

------------------------------------------------------------------------

### 5. What happens when you one-hot encode?

Most algorithms need numbers, so a categorical variable has to become numeric
somehow. One-hot encoding gives each category its own 0/1 column.

In [ ]:
gp_encoded = pd.get_dummies(gp_py, columns=["GP_practice"], dtype=int)

print(gp_py.shape[1])
print(gp_encoded.shape[1])

cols = ["patient_id"] + [c for c in gp_encoded.columns
                         if c.startswith("GP_practice_")]
print(gp_encoded[cols].head(10))

Five columns became nine: the practice column is gone and five indicator
columns have taken its place.

Now try the same thing on the patient identifier:

In [ ]:
gp_ids_encoded = pd.get_dummies(gp_py, columns=["patient_id"], dtype=int)

print({"original": gp_py.shape[1],
       "practice_encoded": gp_encoded.shape[1],
       "id_encoded": gp_ids_encoded.shape[1]})

> **Think about it**
>
> 1. How many columns did each encoding add?
> 2. What do the 0 and the 1 actually mean?
> 3. Why was collapsing the rare practices worth doing *before* encoding?
> 4. Would the patient-identifier columns help predict anything about a patient
>    who is not in this table?
> 5. So: should every categorical variable be one-hot encoded?

------------------------------------------------------------------------